# Drosophila Gene–Gene (STRING) — Relation-Wise KG Triple Construction

## Purpose

This notebook processes **Protein–Protein interaction data** from the STRING database for *Drosophila melanogaster* and transforms it into standardized Gene–Gene relation-wise Knowledge Graph (KG) triples. FlyBase protein IDs are mapped to gene identifiers via gProfiler.

## Pipeline Overview

| Step | Section | Description |
|------|---------|-------------|
| 1 | gProfiler Mapping | Load gProfiler (FlyBase Protein ID → Gene ID + Description) |
| 2 | STRING Loading & Preprocessing | Load raw STRING file, strip species prefix, uppercase protein IDs |
| 3 | KG Triple Construction | Map proteins to genes, add metadata, annotate descriptions |
| 4 | Filter, Validate, Deduplicate & Export | Remove unmapped, validate, deduplicate, export |

## Final Output Schema

| Column | Description |
|--------|-------------|
| `head` | Gene identifier (FlyBase) |
| `relation` | `Gene_Gene` |
| `tail` | Gene identifier (FlyBase) |
| `head_type` | `Gene` |
| `relation_type` | (NaN — no sub-relation) |
| `tail_type` | `Gene` |
| `kg_source` | `STRING` |
| `head_id_is` | `Flybase` |
| `tail_id_is` | `Flybase` |
| `head_detail_name` | Gene description from gProfiler |
| `tail_detail_name` | Gene description from gProfiler |
| `species` | `D.melanogaster` |

## Data Download Instructions

All databases used in this notebook must be downloaded prior to execution.
Please refer to the **central download instructions document** for detailed steps:

📄 **[Download Instructions — Link to be added]**

### Required Files

| File | Source | Description |
|------|--------|-------------|
| `7227.protein.links.detailed.v12.0.txt` | STRING | Raw protein–protein interaction file for Drosophila (tax_id: 7227) |
| `gProfiler_dmelanogaster_*.csv` | gProfiler | FlyBase protein ID to gene ID/description mapping |

---
## Setup

Import required libraries and define all base paths.

In [1]:
import pandas as pd
import numpy as np

In [6]:
# =============================================================================
# BASE PATHS — Update these to match your local directory structure
# =============================================================================
your_path_here = '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/'

# gProfiler mapping file (FlyBase protein ID → gene ID)
GPROFILER_PATH = f'{your_path_here}flybase/Gprofiler_FBpp_2_Gene_Symbol.csv'
# GPROFILER_PATH = f'{your_path_here}flybase/gProfiler_dmelanogaster_4-9-2025_1-52-50 PM.csv'

# Raw STRING protein-protein interaction file for Drosophila
STRING_RAW_PATH = f'{your_path_here}stitch/dmel/7227.protein.links.detailed.v12.0.txt'

# Final output path
OUTPUT_PATH = '/storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/dmel/string_DROSO_GENE_GENE.csv'

In [8]:
# pd.read_csv(GPROFILER_PATH)

---
## 1. Load gProfiler Mapping (FlyBase Protein ID → Gene ID + Description)

gProfiler provides a mapping from FlyBase protein IDs (e.g., `FBPP0070001`) to gene identifiers (via `converted_alias`) and descriptions. Both head and tail proteins will be mapped using this dictionary.

In [9]:
# =============================================================================
# Load gProfiler data and build mapping dictionaries
# =============================================================================

gprofiler_mapping = pd.read_csv(GPROFILER_PATH)
gprofiler_mapping = gprofiler_mapping[~gprofiler_mapping['converted_alias'].isna()]

# Dictionary: FlyBase Protein ID → Gene ID (via converted_alias)
protein_to_gene_dict = dict(zip(gprofiler_mapping['initial_alias'], gprofiler_mapping['converted_alias']))

# Dictionary: Gene ID → Gene Description
gene_to_description_dict = dict(zip(gprofiler_mapping['converted_alias'], gprofiler_mapping['description']))

print(f"gProfiler entries loaded: {len(gprofiler_mapping):,}")
print(f"Protein → Gene dictionary size: {len(protein_to_gene_dict):,}")
print(f"Gene → Description dictionary size: {len(gene_to_description_dict):,}")

gProfiler entries loaded: 10,663
Protein → Gene dictionary size: 10,663
Gene → Description dictionary size: 10,663


---
## 2. Load and Preprocess STRING Protein–Protein Data

Load the raw STRING file, strip the species prefix (`7227.`), rename columns, and uppercase protein IDs (required for matching with gProfiler dictionary keys).

In [10]:
# =============================================================================
# Load raw STRING protein-protein interaction file
# =============================================================================

string_ppi_raw = pd.read_csv(STRING_RAW_PATH, sep='\\s')

# Rename protein columns to Head/Tail
string_ppi_raw = string_ppi_raw.rename(columns={'protein1': 'Head', 'protein2': 'Tail'})

# Strip species prefix '7227.' from both protein columns
string_ppi_raw['Head'] = string_ppi_raw['Head'].str.replace('7227.', '', regex=False)
string_ppi_raw['Tail'] = string_ppi_raw['Tail'].str.replace('7227.', '', regex=False)

# Uppercase protein IDs (required for gProfiler dictionary matching)
string_ppi_raw['Tail'] = string_ppi_raw['Tail'].str.upper()
string_ppi_raw['Head'] = string_ppi_raw['Head'].str.upper()

print(f"Raw STRING interactions loaded: {len(string_ppi_raw):,}")
string_ppi_raw.head()

/tmp/ipykernel_1661027/3996122002.py:5: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  string_ppi_raw = pd.read_csv(STRING_RAW_PATH, sep='\\s')


Raw STRING interactions loaded: 5,033,960


,Head,Tail,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
0,FBPP0070001,FBPP0082651,0,0,0,375,0,0,0,375
1,FBPP0070001,FBPP0078514,0,0,0,379,0,0,0,379
2,FBPP0070001,FBPP0083155,0,0,0,384,0,0,0,384
3,FBPP0070001,FBPP0304379,0,0,0,243,0,0,0,242
4,FBPP0070001,FBPP0076311,0,0,0,174,0,0,0,173


---
## 3. KG Triple Construction — Map Proteins to Genes and Annotate

Map both head and tail protein IDs to gene identifiers via gProfiler, add KG metadata, and annotate with gene descriptions.

In [11]:
# =============================================================================
# Add KG metadata and map protein IDs to gene identifiers
# =============================================================================

gene_gene_df = string_ppi_raw

gene_gene_df['relation_type'] = np.nan
gene_gene_df['kg_source'] = 'STRING'

# Map FlyBase Protein IDs to gene identifiers using gProfiler
gene_gene_df['tail'] = gene_gene_df['Tail'].map(protein_to_gene_dict)
gene_gene_df['head'] = gene_gene_df['Head'].map(protein_to_gene_dict)

# Set identifier namespace
gene_gene_df['head_id_is'] = 'Flybase'
gene_gene_df['tail_id_is'] = 'Flybase'

print(f"Protein IDs mapped to gene identifiers")
print(f"Head mapped: {gene_gene_df['head'].notna().sum():,}")
print(f"Tail mapped: {gene_gene_df['tail'].notna().sum():,}")

Protein IDs mapped to gene identifiers
Head mapped: 3,532,111
Tail mapped: 3,532,111


In [12]:
# =============================================================================
# Map gene descriptions for both head and tail
# =============================================================================

gene_gene_df['tail_detail_name'] = gene_gene_df['tail'].map(gene_to_description_dict)
gene_gene_df['head_detail_name'] = gene_gene_df['head'].map(gene_to_description_dict)

print(f"Descriptions mapped")
print(f"Head descriptions: {gene_gene_df['head_detail_name'].notna().sum():,}")
print(f"Tail descriptions: {gene_gene_df['tail_detail_name'].notna().sum():,}")

Descriptions mapped
Head descriptions: 2,537,091
Tail descriptions: 2,537,091


---
## 4. Filter, Validate, Deduplicate & Export

Filter out entries with unmapped descriptions, add final metadata, validate, deduplicate, and export.

In [13]:
# =============================================================================
# Filter out entries with unmapped descriptions
# =============================================================================

before_count = len(gene_gene_df)
gene_gene_df = gene_gene_df[~gene_gene_df['tail_detail_name'].isna()]
gene_gene_df = gene_gene_df[~gene_gene_df['head_detail_name'].isna()]
after_count = len(gene_gene_df)

print(f"Before filtering: {before_count:,}")
print(f"After filtering: {after_count:,}")
print(f"Removed: {before_count - after_count:,} triples with unmapped genes")

Before filtering: 5,033,960
After filtering: 1,392,012
Removed: 3,641,948 triples with unmapped genes


In [14]:
# =============================================================================
# Add final metadata
# =============================================================================

gene_gene_df['species'] = 'D.melanogaster'
gene_gene_df['relation'] = 'Gene_Gene'
gene_gene_df['head_type'] = 'Gene'
gene_gene_df['tail_type'] = 'Gene'

print(f"Final metadata added. Triples: {len(gene_gene_df):,}")

Final metadata added. Triples: 1,392,012


In [15]:
# =============================================================================
# Data quality validation
# =============================================================================

true_nan_count = gene_gene_df.isna().sum()
string_nan_count = gene_gene_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

nan_summary = pd.DataFrame({
    'NaN_count': true_nan_count,
    "'NAN'_string_count": string_nan_count,
    'Total_NaN_like': true_nan_count + string_nan_count
})

print("NaN validation summary:")
nan_summary

NaN validation summary:


,NaN_count,'NAN'_string_count,Total_NaN_like
Head,0,0,0
Tail,0,0,0
neighborhood,0,0,0
fusion,0,0,0
cooccurence,0,0,0
coexpression,0,0,0
experimental,0,0,0
database,0,0,0
textmining,0,0,0
combined_score,0,0,0


In [16]:
# =============================================================================
# Deduplicate by grouping on (head, relation, tail)
# =============================================================================

group_cols = ['head', 'relation', 'tail']

def merge_sources(x):
    """Merge multiple kg_source values with '::' separator."""
    return '::'.join(sorted(set(x.dropna())))

gene_gene_dedup = gene_gene_df.groupby(group_cols, as_index=False).agg({
    'head_type': 'first',
    'relation_type': 'first',
    'tail_type': 'first',
    'kg_source': merge_sources,
    'head_id_is': 'first',
    'tail_id_is': 'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Triples before deduplication: {len(gene_gene_df):,}")
print(f"Triples after deduplication: {len(gene_gene_dedup):,}")
print(f"Duplicates removed: {len(gene_gene_df) - len(gene_gene_dedup):,}")
gene_gene_dedup.head()

Triples before deduplication: 1,392,012
Triples after deduplication: 1,392,012
Duplicates removed: 0


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,128up,Gene_Gene,26-29-p,Gene,NaN,Gene,STRING,Flybase,Flybase,upstream of RpIII128,26-29kD-proteinase,D.melanogaster
1,128up,Gene_Gene,AIMP1,Gene,NaN,Gene,STRING,Flybase,Flybase,upstream of RpIII128,aaRS-interacting multifunctional protein 1,D.melanogaster
2,128up,Gene_Gene,AIMP2,Gene,NaN,Gene,STRING,Flybase,Flybase,upstream of RpIII128,aaRS-interacting multifunctional protein 2,D.melanogaster
3,128up,Gene_Gene,AP-1mu,Gene,NaN,Gene,STRING,Flybase,Flybase,upstream of RpIII128,"Adaptor Protein complex 1, mu subunit",D.melanogaster
4,128up,Gene_Gene,AP-2mu,Gene,NaN,Gene,STRING,Flybase,Flybase,upstream of RpIII128,"Adaptor Protein complex 2, mu subunit",D.melanogaster


In [17]:
# =============================================================================
# Export final deduplicated Gene-Gene KG triples
# =============================================================================

gene_gene_dedup.to_csv(OUTPUT_PATH, index=None)
print(f"Final output saved to: {OUTPUT_PATH}")
print(f"Total triples exported: {len(gene_gene_dedup):,}")

Final output saved to: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/string_DROSO_GENE_GENE.csv
Total triples exported: 1,392,012


---
## Summary

This notebook processed STRING Protein–Protein interaction data for *D. melanogaster* into standardized Gene–Gene KG triples:

1. **gProfiler setup** — Loaded FlyBase protein → gene ID mappings and descriptions
2. **STRING preprocessing** — Loaded raw interactions, stripped `7227.` prefix, uppercased protein IDs
3. **KG triple construction** — Mapped both head and tail proteins to gene IDs via gProfiler
4. **Annotation & filtering** — Added gene descriptions, filtered unmapped entries
5. **Deduplication** — Grouped by (head, relation, tail) for unique final triples

### Output

| File | Description |
|------|-------------|
| `string_DROSO_GENE_GENE.csv` | Final: deduplicated relation-wise KG triples |